In [ ]:
#Shortage prediction model training

In [ ]:
import pandas as pd 
df=pd.read_pickle('C:/Users/Rishj/OneDrive/Desktop/PROJECTS/supply_chain_intelligence/data/processed/shortage_engineered_dataset.pkl')
print(df.shape)

(172762, 54)


In [ ]:
from sklearn.model_selection import train_test_split

shortage_feature_cols = ['product_demand_mean', 'product_demand_std', 'qty_deviation', 'product_order_history_count']

# Sort by actual order date (chronological across all products) before splitting
df = df.sort_values('order date (DateOrders)').reset_index(drop=True)

X_shortage = df[shortage_feature_cols]
y_shortage = df['shortage_risk']

split_index = int(len(df) * 0.8)
X_train_s, X_test_s = X_shortage[:split_index], X_shortage[split_index:]
y_train_s, y_test_s = y_shortage[:split_index], y_shortage[split_index:]

print(X_train_s.shape, X_test_s.shape)
print(y_train_s.value_counts())
print(y_test_s.value_counts())

(138209, 4) (34553, 4)
shortage_risk
0    134651
1      3558
Name: count, dtype: int64
shortage_risk
0    33069
1     1484
Name: count, dtype: int64


In [ ]:
#train the baseline model whenever you are. Given the class imbalance we just saw, let's use Logistic Regression with class_weight='balanced'
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

shortage_model = LogisticRegression(max_iter=1000, class_weight='balanced')
shortage_model.fit(X_train_s, y_train_s)

y_pred_s = shortage_model.predict(X_test_s)

print(confusion_matrix(y_test_s, y_pred_s))
print(classification_report(y_test_s, y_pred_s))

[[25896  7173]
 [  628   856]]
              precision    recall  f1-score   support

           0       0.98      0.78      0.87     33069
           1       0.11      0.58      0.18      1484

    accuracy                           0.77     34553
   macro avg       0.54      0.68      0.52     34553
weighted avg       0.94      0.77      0.84     34553



In [ ]:
#actual predicted probabilities (not just 0/1 predictions)
# Probability of class 1 (shortage risk) for each test row
y_proba_s = shortage_model.predict_proba(X_test_s)[:, 1]
print(y_proba_s[:10])  # peek at a few probability values

[0.75960438 0.3108787  0.2598036  0.13173291 0.23268528 0.23740895
 0.72344026 0.089055   0.14937011 0.0994217 ]


In [ ]:
#Try a few different thresholds and compare precision/recall
from sklearn.metrics import precision_score, recall_score, f1_score
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_thresh = (y_proba_s >= threshold).astype(int)
    precision = precision_score(y_test_s, y_pred_thresh)
    recall = recall_score(y_test_s, y_pred_thresh)
    f1 = f1_score(y_test_s, y_pred_thresh)
    print(f"Threshold: {threshold} | Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}")

Threshold: 0.3 | Precision: 0.090 | Recall: 0.577 | F1: 0.155
Threshold: 0.4 | Precision: 0.102 | Recall: 0.577 | F1: 0.173
Threshold: 0.5 | Precision: 0.107 | Recall: 0.577 | F1: 0.180
Threshold: 0.6 | Precision: 0.107 | Recall: 0.555 | F1: 0.180
Threshold: 0.7 | Precision: 0.072 | Recall: 0.345 | F1: 0.119


In [ ]:
#training the Random Forest model for shortage prediction
from sklearn.ensemble import RandomForestClassifier

rf_shortage_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=50,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_shortage_model.fit(X_train_s, y_train_s)

# Train vs test accuracy check (overfitting check, like before)
train_pred_rf_s = rf_shortage_model.predict(X_train_s)
test_pred_rf_s = rf_shortage_model.predict(X_test_s)

print("Train accuracy:", accuracy_score(y_train_s, train_pred_rf_s))
print("Test accuracy:", accuracy_score(y_test_s, test_pred_rf_s))

print(confusion_matrix(y_test_s, test_pred_rf_s))
print(classification_report(y_test_s, test_pred_rf_s))

Train accuracy: 0.895173252103698
Test accuracy: 0.8169768182212832
[[27957  5112]
 [ 1212   272]]
              precision    recall  f1-score   support

           0       0.96      0.85      0.90     33069
           1       0.05      0.18      0.08      1484

    accuracy                           0.82     34553
   macro avg       0.50      0.51      0.49     34553
weighted avg       0.92      0.82      0.86     34553



In [ ]:
# scale_pos_weight ≈ (count of negative class) / (count of positive class)
from xgboost import XGBClassifier

scale_pos_weight = (y_train_s == 0).sum() / (y_train_s == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

xgb_shortage_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)
xgb_shortage_model.fit(X_train_s, y_train_s)

train_pred_xgb_s = xgb_shortage_model.predict(X_train_s)
test_pred_xgb_s = xgb_shortage_model.predict(X_test_s)

print("Train accuracy:", accuracy_score(y_train_s, train_pred_xgb_s))
print("Test accuracy:", accuracy_score(y_test_s, test_pred_xgb_s))

print(confusion_matrix(y_test_s, test_pred_xgb_s))
print(classification_report(y_test_s, test_pred_xgb_s))

scale_pos_weight: 37.844575604272066
Train accuracy: 0.9311477544877685
Test accuracy: 0.8899661389749081
[[30483  2586]
 [ 1216   268]]
              precision    recall  f1-score   support

           0       0.96      0.92      0.94     33069
           1       0.09      0.18      0.12      1484

    accuracy                           0.89     34553
   macro avg       0.53      0.55      0.53     34553
weighted avg       0.92      0.89      0.91     34553



In [ ]:
import joblib
joblib.dump(shortage_model, 'shortage_prediction_model.joblib')

['shortage_prediction_model.joblib']